> Projeto Desenvolve <br>
Programação Intermediária com Python <br>
Profa. Camila Laranjeira (mila@projetodesenvolve.com.br) <br>

# 3.14 - ORM

## Exercícios

#### Q1. Conhecendo os dados
Baixe o seguinte csv onde iremos trabalhar. Ele contém informações sobre salários de profissionais de dados de uma empresa hipotética entre 2009 e 2016
* https://github.com/camilalaranjeira/python-intermediario/blob/main/salaries.csv

Suas colunas, descritas na [página do Kaggle que contém o dataset](https://www.kaggle.com/datasets/krishujeniya/salary-prediction-of-data-professions?resource=download), são:
* FIRST NAME: Primeiro nome do profissional de dados (String)
* LAST NAME: Sobrenome do profissional de dados (String)
* SEX: Gênero do profissional de dados (String: 'F' para Feminino, 'M' para Masculino)
* DOJ (Date of Joining): A data em que o profissional de dados ingressou na empresa (Data no formato MM/DD/AAAA)
* CURRENT DATE: A data atual ou a data de referência dos dados (Data no formato MM/DD/AAAA)
* DESIGNATION: O cargo ou designação do profissional de dados (String: ex., Analista, Analista Sênior, Gerente)
* AGE: Idade do profissional de dados (Integer)
* SALARY: Salário anual do profissional de dados (Float)
* UNIT: Unidade de negócios ou departamento em que o profissional de dados trabalha (String: ex., TI, Finanças, Marketing)
* LEAVES USED: Número de licenças utilizadas pelo profissional de dados (Integer)
* LEAVES REMAINING: Número de licenças restantes para o profissional de dados (Integer)
* RATINGS: Avaliações de desempenho do profissional de dados (Float)
* PAST EXP: Experiência de trabalho anterior em anos antes de ingressar na empresa atual (Float)

Na célula a seguir, **carregue os dados do CSV e dê uma olhada neles antes de seguir**.

In [ ]:
import pandas as pd

df = pd.read_csv("salaries.csv")

df = df.rename(columns={
    "FIRST NAME": "first_name",
    "LAST NAME": "last_name",
    "SEX": "sex",
    "DOJ": "doj",
    "CURRENT DATE": "current_date",
    "DESIGNATION": "designation",
    "AGE": "age",
    "SALARY": "salary",
    "UNIT": "unit",
    "LEAVES USED": "leaves_used",
    "LEAVES REMAINING": "leaves_remaining",
    "RATINGS": "ratings",
    "PAST EXP": "past_exp"
})


df["doj"] = pd.to_datetime(df["doj"], errors="coerce")
df["current_date"] = pd.to_datetime(df["current_date"], errors="coerce")


print("Quantidade de linhas e colunas:")
print(df.shape)

print("\nPrimeiras 5 linhas:")
print(df.head())

print("\nInformações gerais:")
df.info()

print("\nEstatísticas das colunas numéricas:")
print(df.describe())

print("\nValores únicos de SEX:")
print(df["sex"].unique())

print("\nValores únicos de DESIGNATION:")
print(df["designation"].unique())

print("\nValores únicos de UNIT:")
print(df["unit"].unique())

# Verifica valores ausentes
print("\nValores ausentes por coluna:")
print(df.isnull().sum())

#### Q2. Modelando os dados
Você deve **criar um ORM com SQLAlchemy capaz de comportar os dados dessa base**.

* Crie um campo de chave primária `ID`, que deve ser incrementado automaticamente
* Os campos SEX, DESIGNATION e UNIT devem ser definidos como classes `Enum` com os possíveis valores (consulte os valores únicos dessas colunas)
* Para os outros campos, consulte os tipos de dados informados na descrição acima

In [ ]:
from enum import Enum

from sqlalchemy import (
    Column,
    Integer,
    String,
    Float,
    Date,
    Enum as SQLEnum
)

from sqlalchemy.orm import declarative_base

Base = declarative_base()


# Enums

class Sex(Enum):
    F = "F"
    M = "M"


class Designation(Enum):
    Analyst = "Analyst"
    Senior_Analyst = "Senior Analyst"
    Associate = "Associate"
    Manager = "Manager"
    Senior_Manager = "Senior Manager"
    Director = "Director"


class Unit(Enum):
    Finance = "Finance"
    IT = "IT"
    Web = "Web"
    Operations = "Operations"
    Marketing = "Marketing"
    Management = "Management"


# Modelo ORM

class Employee(Base):
    __tablename__ = "employees"

    id = Column(Integer, primary_key=True, autoincrement=True)

    first_name = Column(String(100))
    last_name = Column(String(100))

    sex = Column(SQLEnum(Sex))

    doj = Column(Date)
    current_date = Column(Date)

    designation = Column(SQLEnum(Designation))

    age = Column(Integer)
    salary = Column(Float)

    unit = Column(SQLEnum(Unit))

    leaves_used = Column(Integer)
    leaves_remaining = Column(Integer)

    ratings = Column(Float)

    past_exp = Column(Float)

#### Q3. Estabelecendo uma conexão

Usando o método `create_engine` do SQLAlchemy, crie uma conexão com um novo banco de dados SQLite chamado `salarios`.

In [ ]:
from sqlalchemy import create_engine

# Cria a conexão com o banco SQLite
engine = create_engine("sqlite:///salarios.db")

print("Conexão criada com sucesso!")

#### Q4. Criando as tabelas
Crie as tabelas da questão Q2 no banco `salarios`.

In [ ]:
# Cria todas as tabelas definidas nos modelos ORM
Base.metadata.create_all(engine)

print("Tabelas criadas com sucesso!")

#### Q5. Populando

Usando o método `to_sql` da biblioteca Pandas (veja [a documentação](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.to_sql.html)), popule o banco `salarios` com os dados do csv que você carregou na questão Q1.
* Lembre-se de definir o parâmetro `if_exists='append'` para que as tabelas não sejam dropadas e recriadas.

In [ ]:
# Insere os dados do DataFrame na tabela employees
df.to_sql(
    "employees",
    con=engine,
    if_exists="append",
    index=False
)

print("Dados inseridos com sucesso!")

#### Q6. Consultas SQL vs ORM

Agrupe os dados por DESIGNATION e selecione o mínimo, máximo e a média dos salários (SALARY) divididos por 12. Já que o atributo SALARY é anual, dividir por 12 nos mostrará os valores mensais.

Assumindo que a variável que armazena a sua conexão se chama `engine`, você deve realizar a query acima de três formas:
* Executando a query SQL através de uma instância de conexão retornada pelo método `engine.connect()`
* Executando a query SQL com o método `read_sql_query` do Pandas (veja [a documentação](https://pandas.pydata.org/docs/reference/api/pandas.read_sql_query.html)). Você usará mesma instância `engine.connect()` como um dos parâmetros.
* Executando uma query criada com o módulo `select` do SQLAlchemy. Sua execução deve ser feita através de um objeto `Session` do módulo `orm` do SQLAlchemy (`Session(engine)`).


In [ ]:
from sqlalchemy import text

query = """
SELECT
    designation,
    MIN(salary) / 12 AS salario_minimo,
    MAX(salary) / 12 AS salario_maximo,
    AVG(salary) / 12 AS salario_medio
FROM employees
GROUP BY designation
"""

with engine.connect() as conn:
    resultado = conn.execute(text(query))

    for linha in resultado:
        print(linha)

In [ ]:
import pandas as pd

query = """
SELECT
    designation,
    MIN(salary) / 12 AS salario_minimo,
    MAX(salary) / 12 AS salario_maximo,
    AVG(salary) / 12 AS salario_medio
FROM employees
GROUP BY designation
"""

with engine.connect() as conn:
    resultado = pd.read_sql_query(query, conn)

print(resultado)

In [ ]:
from sqlalchemy import func
from sqlalchemy.orm import Session

with Session(engine) as session:

    resultado = session.query(
        Employee.designation,
        (func.min(Employee.salary) / 12).label("salario_minimo"),
        (func.max(Employee.salary) / 12).label("salario_maximo"),
        (func.avg(Employee.salary) / 12).label("salario_medio")
    ).group_by(
        Employee.designation
    ).all()

    for linha in resultado:
        print(linha)